# Diabetic Retinopathy baseline - thin Colab interface

This notebook now does almost nothing itself. Every real step is a tested function in
`colab/pipeline.py`, so the same work also runs from a terminal and from tests:

| Terminal command | Notebook cell |
| --- | --- |
| `python -m colab.pipeline preflight` | `# 2` |
| `python -m colab.pipeline canary` | `# 3` (opt-in) |
| `python -m colab.pipeline train --confirm` | `# 4` (opt-in) |
| `python -m colab.pipeline train --confirm --resume` | `# 4` again, after a disconnect |
| `python -m colab.pipeline evaluate --confirm` | `# 5` (opt-in) |
| `python -m colab.pipeline status` | `# 6` |

**Only three actions are normally needed:** run `# 2`, then `# 4` (set `RUN_TRAINING = True`),
then `# 5` (set `RUN_FINAL_EVALUATION = True`) once training finished. Cell `# 1` only locates the
project and must run once; after that, cells never depend on each other's *results* (the training
cell re-runs preflight itself if it was skipped), so re-running or skipping a cell cannot produce a
`NameError` and cannot start training against an unverified environment.

**Status:** the five-band primary mapping is provisional and unsigned (see
`audit_output/pretraining_audit/LABEL_POLICY_DRAFT.md`). Nothing here is a clinical result.
The audited split, the label mapping, and the audit artifacts are only ever read.

## 1 - Setup

Locates the project (never an unbounded scan), puts it on `sys.path`, and imports the tested
pipeline module. Safe to run first, or after any other cell, in any order.

In [ ]:
# 1. Setup: locate the project and import the tested pipeline module.
import sys
from pathlib import Path

# Set this only if discovery fails, e.g. "/content/drive/MyDrive/major_project".
OVERRIDE_PROJECT_ROOT = ""
# The environment variable DR_PROJECT_ROOT works too and takes the same value.

# Legacy consoles (and some Colab log streams) raise on characters such as the
# label-policy warning sign. Report as "?" instead of killing the cell.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(errors="replace")
    except Exception:
        pass

# Mount Drive first so runs, checkpoints, and logs survive a Colab disconnect.
# Wrapped: outside Colab this is skipped and the output dir simply stays local.
try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except Exception as _exc:
    print(f"Drive not mounted (fine outside Colab): {type(_exc).__name__}")

for _candidate in (Path.cwd(), Path("/content/major_project")):
    if (_candidate / "colab" / "pipeline.py").is_file() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

import colab.pipeline as pipeline

PROJECT_ROOT = pipeline.discover_project(OVERRIDE_PROJECT_ROOT or None)
for _entry in (str(PROJECT_ROOT), str(PROJECT_ROOT / "colab")):
    if _entry not in sys.path:
        sys.path.insert(0, _entry)

print("project root :", PROJECT_ROOT)
print("split dir    :", PROJECT_ROOT / "provisional_primary_split")
print("output dir   :", pipeline.default_output_dir(PROJECT_ROOT))
print("")
print("Terminal equivalent: python -m colab.pipeline preflight")

## 2 - Preflight (run this first)

Checks the whole environment **and** every artifact in one pass and reports **all** failures
together, each with a `->` fix hint:

- project markers, Python and dependency versions, device/CUDA state;
- the pinned split, via the strict (unchanged) validator: 14,201 rows, 9,924 / 2,133 / 2,144;
- the corpus root, **all 14,201 image paths**, and a decode sample from every (partition, class);
- group leakage, the eligibility manifest, the ImageNet weight cache, and output-dir writability;
- the pytest smoke suite, with its real exit code.

If anything fails, the cell stops here. That is the whole point: no training, no canary, no
mystery traceback three cells later.

In [ ]:
# 2. Preflight: all environment and artifact checks, reported together.
RUN_PREFLIGHT = True

PREFLIGHT_REPORT = None
if not RUN_PREFLIGHT:
    print("preflight skipped (not recommended: training would rediscover the same failures later)")
else:
    PREFLIGHT_REPORT = pipeline.preflight(PROJECT_ROOT, run_smoke=True)
    pipeline.print_preflight(PREFLIGHT_REPORT)
    if not PREFLIGHT_REPORT["ok"]:
        raise SystemExit(
            "PREFLIGHT FAILED. Fix each FAIL line above (every one has a -> hint), then re-run "
            "this cell. Do not run the training cell until this prints PREFLIGHT OK."
        )

## 3 - Canary (optional, off by default)

Builds the 75-row derived split at `<output_dir>/canary_split/provisional_primary_split.csv`
(50 train / 25 validation / 0 test), validates it with the **canary-specific** validator
(all five bands in both partitions, per-band counts, no duplicated paths, no group leakage,
every image path present, one decoded image per band - all failing closed), and optionally trains
it for 2 epochs.

The strict full-split validator is untouched: it still requires the audited 14,201 rows with all
three partitions, so a canary can never be passed off as the real split.

In [ ]:
# 3. Canary: build + validate the 75-row derived split (optionally train it).
RUN_CANARY = False
CANARY_TRAIN = False              # set True to actually run 2 epochs on the canary
CANARY_ALLOW_WEIGHT_DOWNLOAD = False

try:
    PROJECT_ROOT
except NameError:  # cell #1 was skipped
    PROJECT_ROOT = pipeline.discover_project()
OUTPUT_DIR = pipeline.default_output_dir(PROJECT_ROOT)
CANARY_RESULT = None
if not RUN_CANARY:
    print("canary skipped. Set RUN_CANARY = True to prove the pipeline on a tiny subset.")
else:
    CANARY_RESULT = pipeline.canary_run(
        project_root=PROJECT_ROOT,
        train=CANARY_TRAIN,
        confirm=CANARY_TRAIN,
        allow_weight_download=CANARY_ALLOW_WEIGHT_DOWNLOAD,
        epochs=2,
    )
    print("canary split:", CANARY_RESULT["rows"], "rows", CANARY_RESULT["partition_counts"])

## 4 - Train, or resume after a disconnect

One cell for both. `RESUME = "auto"` means: continue the run already in `OUTPUT_DIR` if it has a
usable checkpoint, otherwise start a new one. Re-running this cell after a Colab disconnect picks
up from the **last complete epoch**, restoring model weights, optimizer state, the epoch
histories, the best-validation bookkeeping, and the early-stopping counter.

Guards that still have to pass:

- `RUN_TRAINING = True` (this cell's opt-in);
- preflight OK (re-run automatically here if cell `# 2` was not run);
- `ALLOW_WEIGHT_DOWNLOAD = True` when the ImageNet weights are not already cached - the run
  refuses instead of downloading silently;
- the held-out test partition is never loaded here.

Logs and every artifact land in `OUTPUT_DIR` (`logs/`, `checkpoints/epoch_XXXX/`,
`best_checkpoint/`, `run_manifest.json`, `resume_state.json`). On Colab, `OUTPUT_DIR` defaults to
Drive so a disconnect does not lose the run.

In [ ]:
# 4. Train or resume (opt-in, guarded). Re-run after a disconnect to continue.
RUN_TRAINING = False
RESUME = "auto"                 # "auto" | True (must resume) | False (always start fresh)
ALLOW_WEIGHT_DOWNLOAD = False
OVERWRITE = False               # True discards the previous run in OUTPUT_DIR
EPOCHS = 10

try:
    PROJECT_ROOT
except NameError:  # cell #1 was skipped
    PROJECT_ROOT = pipeline.discover_project()
OUTPUT_DIR = pipeline.default_output_dir(PROJECT_ROOT)
TRAIN_RESULT = None

if not RUN_TRAINING:
    print("training skipped. Set RUN_TRAINING = True to start or resume.")
else:
    # Re-derive preflight here so this cell works even if cell #2 was never run.
    try:
        _report = PREFLIGHT_REPORT
    except NameError:  # cell #2 was skipped
        _report = None
    if not _report or not _report.get("ok"):
        _report = pipeline.preflight(PROJECT_ROOT, run_smoke=True)
        pipeline.print_preflight(_report)
        if not _report["ok"]:
            raise SystemExit("training blocked: preflight failed (see the FAIL lines above).")

    TRAIN_RESULT = pipeline.train_run(
        project_root=PROJECT_ROOT,
        output_dir=OUTPUT_DIR,
        epochs=EPOCHS,
        resume=RESUME,
        overwrite=OVERWRITE,
        allow_weight_download=ALLOW_WEIGHT_DOWNLOAD,
        confirm=True,
        skip_preflight=True,   # already checked immediately above
    )
    print("")
    print("log file:", TRAIN_RESULT["log_path"])
    print("best epoch:", TRAIN_RESULT["best_epoch"], "| best val loss:", TRAIN_RESULT["best_val_loss"])
    print("next: set RUN_FINAL_EVALUATION = True in cell #5 to score the held-out test set.")

## 5 - Final test evaluation (once, opt-in)

The only step that touches the held-out test partition. It refuses unless the run's manifest says
`completed` and a validation-selected `best_checkpoint/model.pt` exists, then reports accuracy,
macro precision/recall/F1, per-class metrics, and the confusion matrix.

Do not tune anything against this number: re-running it after changes would invalidate it.

In [ ]:
# 5. Final test evaluation: held-out partition, once.
RUN_FINAL_EVALUATION = False

try:
    PROJECT_ROOT
except NameError:  # cell #1 was skipped
    PROJECT_ROOT = pipeline.discover_project()
OUTPUT_DIR = pipeline.default_output_dir(PROJECT_ROOT)
EVAL_RESULT = None

if not RUN_FINAL_EVALUATION:
    print("final evaluation skipped. Set RUN_FINAL_EVALUATION = True to score the test set.")
else:
    EVAL_RESULT = pipeline.evaluate_run(
        project_root=PROJECT_ROOT,
        output_dir=OUTPUT_DIR,
        confirm=True,
    )
    print("")
    print("test metrics saved to:", EVAL_RESULT["output_path"])

## 6 - Verify the artifacts

Re-checks the output directory file by file (manifest `completed`, `run_summary.json`, best
checkpoint, per-epoch checkpoints, no `TRAINING_INCOMPLETE.txt`, test metrics when requested) and
prints whether the run is resumable.

In [ ]:
# 6. Verify the artifacts and report resume state.
import colab.setup_helpers as sh

try:
    PROJECT_ROOT
except NameError:  # cell #1 was skipped
    PROJECT_ROOT = pipeline.discover_project()
OUTPUT_DIR = pipeline.default_output_dir(PROJECT_ROOT)
VERIFICATION = sh.verify_output_run(OUTPUT_DIR)
pipeline.print_status(pipeline.run_state(OUTPUT_DIR))

print("")
print("=== artifact verification ===")
for item in VERIFICATION["items"]:
    flag = "OK" if item["ok"] else ("FAIL" if item.get("required") else "missing")
    print(f"  [{flag}] {item['name']}: {item['detail']}")
print("")
print("VERIFICATION_OK:", VERIFICATION["ok"])
print(sh.load_label_policy_excerpt(PROJECT_ROOT, max_chars=600))